# Two tests of what is new

**Test A: does it matter who writes the new answers?** Large models (7B: Qwen2.5-7B, OLMo-2-7B,
Mistral-7B v0.3) write answers to the same 2,264 training questions as our small models did. The
same teacher marks them, and a student is trained exactly like `kd17m`, with the same number of new
answers (`kdlarge17m`). Both are then scored on RAGTruth (answers by large models) and on **PsiloQA**
(answers by 24 models from 135M to 9B, grouped by size; we never train on it). If our idea is right,
`kd17m` should do better on answers written by small models.

**Test B: our small teacher against a large language model as the labeller (the PsiloQA way).**
A free open model, Qwen2.5-7B-Instruct (4-bit, on the Colab GPU), lists the unsupported words in the
same small-model answers. PsiloQA used GPT-4o, which costs money through OpenAI's API; a 7B model
that is 18 times larger than our teacher answers the same question for free. A student learns from
its labels (`llm17m`). Because the LLM gives yes/no labels, we also train a student on our teacher's
labels rounded to yes/no (`kdhard17m`). The LLM is also scored against people on 300 RAGTruth test
answers.

**Everything in this notebook is free**: it only uses the Colab GPU, like the earlier notebooks.

| Part | Needs | Time on a T4 (rough) |
|---|---|---|
| Setup | internet | 3 min |
| A1. The open LLM labels 300 test answers | GPU | 10-15 min |
| A2. The open LLM labels the 6,719 small-model answers | GPU | about 2 hours |
| B1. Large models write answers (3 models, 4-bit) | GPU | 2-3 hours |
| B2. Teacher marks them | GPU | 10 min |
| B3. Train and score the 17M students | GPU | 1-1.5 hours |
| B4. PsiloQA scores and the comparison table | GPU | 20-30 min |
| B5. (optional) the same at 68M | GPU | 3-4 hours |

That is 6-7 hours of GPU for the 17M results, so plan two sessions (for example A1, A2 and B1 on
one day, the rest on the next). Everything is saved to `MyDrive/ondevice_hallu/work`; finished
parts are skipped, so after a disconnect run the setup cells again and continue with the step that
was running. It needs the results of `distill.ipynb` in the same folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, shutil
from pathlib import Path
from IPython.display import Markdown, display

REPO = 'https://github.com/gbirhanu/ondevice-hallu.git'
CODE = Path('/content/ondevice_hallu')
DRIVE = Path('/content/drive/MyDrive/ondevice_hallu')
WORK = DRIVE / 'work'
DATA = Path('/content/RAGTruth')

def sh(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    print(result.stdout[-3000:], flush=True)
    return result.returncode

if (CODE / '.git').exists():
    sh(['git', '-C', str(CODE), 'pull', '--ff-only'])
else:
    assert sh(['git', 'clone', REPO, str(CODE)]) == 0, f'Cannot clone {REPO}: push the code first.'
if not DATA.exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate', 'bitsandbytes', 'datasets'])
need = ['results/kd17m_training.json', 'preds/kd17m_test.json.gz', 'preds/ld_large_train.json.gz']
missing = [n for n in need if not (WORK / n).exists()]
print('Files from distill.ipynb missing:', missing or 'none (good)')

In [ ]:
def run(*arguments):
    """Runs one run.py command and shows its output as it comes. Returns True when it worked."""
    command = [sys.executable, '-u', str(CODE / 'run.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false'})
    for line in process.stdout:
        if not any(x in line for x in ('Loading checkpoint shards', 'Warning', 'Writing model shards', 'Loading weights')):
            print(line, end='', flush=True)
    return process.wait() == 0

def free_disk():
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for d in hub.glob('models--*'):
        shutil.rmtree(d, ignore_errors=True)

SMALL = ['qwen2.5-0.5b', 'qwen2.5-1.5b', 'smollm2-1.7b']
LARGE = ['qwen2.5-7b', 'olmo2-7b', 'mistral-7b-v0.3']
TEACHER = 'ld_large'

## Token (optional) and the labeller

`HF_TOKEN` in the **Secrets** panel (key icon on the left) is only needed for Mistral-7B v0.3, whose
licence must be accepted on its Hugging Face page; without it, the two other large models are used.
No paid key is needed. (Only if you ever have API credit: set `LLM` to the commented line to use a
paid model such as GPT-4o instead of the free one.)

In [ ]:
LABELLER = 'qwen7b'                      # our name for this labeller (used in file names)
LLM = ['--name', LABELLER, '--local', 'qwen2.5-7b']          # free: Qwen2.5-7B-Instruct on this GPU
# Paid alternative (needs API credit; NOT included in a ChatGPT subscription):
# LABELLER = 'gpt4o'; LLM = ['--name', LABELLER, '--llm-model', 'gpt-4o', '--api-key-env', 'OPENAI_API_KEY']

from google.colab import userdata
for name in ('HF_TOKEN', 'OPENAI_API_KEY'):
    try:
        os.environ[name] = userdata.get(name)
        print(f'{name}: found')
    except Exception:
        print(f'{name}: not found' + (' (not needed)' if name == 'OPENAI_API_KEY' else ''))
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'

## Part A1: how good is the LLM as a labeller? (300 RAGTruth test answers)

The same 300 answers are also scored for our teacher, so the two labellers can be compared directly.

In [ ]:
assert run('llmlabel', *LLM, '--on', 'test', '--limit', 300), 'Labelling failed; see the message above.'
r = json.loads((WORK / 'results' / f'llmlabel_{LABELLER}_test.json').read_text())
print(json.dumps({k: r[k] for k in r if k in ('labelled', 'unusable_replies', 'labeller_vs_people',
                  'teacher_vs_people_same_answers', 'labeller_minus_teacher_ci95')}, indent=1))

## Part A2: the LLM labels the small models' new answers

In [ ]:
assert run('llmlabel', *LLM, '--on', 'extra', '--generators', *SMALL), 'Labelling failed.'
free_disk()

## Part B1: large models write answers to the same questions (T4 GPU)

In [ ]:
for g in LARGE:
    if not run('generate', '--generator', g):
        print(f'--> {g} did not finish; the comparison uses the large models that did.')
    free_disk()

## Part B2: the teacher marks the large models' answers

In [ ]:
assert run('teach', '--name', TEACHER), 'The teacher step failed.'
free_disk()

## Part B3: train and score the 17M students

All start from Ettin-17M with the same settings as `kd17m` (5 epochs, learning rate 2e-5). The small
and the large arm get the same number of new answers.

In [ ]:
def kept(gens):
    return {g: json.loads((WORK / 'results' / f'extra_{g}.json').read_text())['kept']
            for g in gens if (WORK / 'results' / f'extra_{g}.json').exists()}

small, large = kept(SMALL), kept(LARGE)
print('new answers, small models:', small, '\nnew answers, large models:', large)
N = min(sum(small.values()), sum(large.values()))
SMALL_RUN = 'kd{size}' if sum(small.values()) <= N * 1.02 else 'kd{size}-n%d' % N   # same number of new answers

def train_and_score(name, size, *options, epochs=5):
    ok = run('train', '--name', name, '--base', f'jhu-clsp/ettin-encoder-{size}', '--epochs', epochs,
             '--lr', 2e-5, '--teacher', TEACHER, *options)
    return ok and run('score', '--name', name)

def students(size):
    if SMALL_RUN.format(size=size) != f'kd{size}':
        train_and_score(SMALL_RUN.format(size=size), size, '--extra', *small, '--extra-max', N)
    if large:
        train_and_score(f'kdlarge{size}', size, '--extra', *large, '--extra-max', N)
    train_and_score(f'kdhard{size}', size, '--extra', *small, '--extra-hard')
    if (WORK / 'preds' / f'{LABELLER}_extra_{SMALL[0]}.json.gz').exists():
        train_and_score(f'llm{size}', size, '--extra', *small, '--extra-teacher', LABELLER)
    else:
        print('No LLM labels yet (Part A2), so the LLM-labelled student is skipped.')

students('17m')

## Part B4: PsiloQA by answer-model size, and the comparison table

In [ ]:
def report(size):
    run('psilo')
    display(Markdown((WORK / 'results' / 'psilo.md').read_text()))
    s = SMALL_RUN.format(size=size)
    pairs = [f'{s}:kdlarge{size}', f'kd{size}:kdhard{size}', f'kdhard{size}:llm{size}', f'kd{size}:llm{size}']
    run('compare', '--pairs', *pairs)
    display(Markdown((WORK / 'results' / 'compare.md').read_text()))

report('17m')

**How to read it.**

- **Test A, row `kd17m → kdlarge17m`:** negative numbers in the PsiloQA *up to 2B* column mean the
  student trained on small-model answers catches small models' mistakes better. That is the finding
  we hope for. On RAGTruth (answers by large models) large-model answers may help more; that would
  fit the same idea: match the training answers to the models you will check.
- **Test B, rows with `llm17m`:** compare `kdhard17m → llm17m` (both trained on yes/no labels; only
  the labeller differs) and `kd17m → llm17m` (our full method against the PsiloQA way). The A1
  output above says how each labeller scores against people.
- `kd17m → kdhard17m`: what soft (probability) labels add over yes/no labels.

## Part B5 (optional): the same at 68M

In [ ]:
students('68m')
report('68m')